In [ ]:
import os
from os import path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import numpy as np
import pickle as pkl

from macaquethings.data_util.load_data import load_inter_area_fit_chunks
from macaquethings.plotting.default_styles import *
from ephyslib.connectivity import check_run_integrity

from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import matplotlib.ticker as tkr
import seaborn as sns

figure_style(font_size=6)  # set consistent plotting defaults for all figs
%matplotlib widget

In [ ]:
rundirs_F_v4_it = [
    "../../results/inter_area/monkeyF_v4_target_array_9_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyF_v4_target_array_10_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyF_v4_target_array_11_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyF_v4_target_array_12_stride_2_ridgecv_lfp_threefold",    
    "../../results/inter_area/monkeyF_v4_target_array_13_stride_2_ridgecv_lfp_threefold",    
]

rundirs_F_v1_v4 = [
    "../../results/inter_area/monkeyF_v1_target_array_14_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyF_v1_target_array_15_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyF_v1_target_array_16_stride_2_ridgecv_lfp_threefold",
]

rundirs_F_v1_it = [
    "../../results/inter_area/monkeyF_v1_target_array_9_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyF_v1_target_array_10_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyF_v1_target_array_11_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyF_v1_target_array_12_stride_2_ridgecv_lfp_threefold",    
    "../../results/inter_area/monkeyF_v1_target_array_13_stride_2_ridgecv_lfp_threefold",        
]

rundirs_N_v4_it = [
    "../../results/inter_area/monkeyN_v4_target_array_13_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyN_v4_target_array_14_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyN_v4_target_array_15_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyN_v4_target_array_16_stride_2_ridgecv_lfp_threefold",
]

rundirs_N_v1_v4 = [
    "../../results/inter_area/monkeyN_v1_target_array_9_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyN_v1_target_array_10_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyN_v1_target_array_11_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyN_v1_target_array_12_stride_2_ridgecv_lfp_threefold",   
]
rundirs_N_v1_it = [
    "../../results/inter_area/monkeyN_v1_target_array_13_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyN_v1_target_array_14_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyN_v1_target_array_15_stride_2_ridgecv_lfp_threefold",
    "../../results/inter_area/monkeyN_v1_target_array_16_stride_2_ridgecv_lfp_threefold",
]

rundirs_F_v4_it_baseline = [
    "../../results/inter_area/monkeyF_v4_target_array_9_stride_2_ridgecv_lfp_threefold_baseline-1",
    "../../results/inter_area/monkeyF_v4_target_array_10_stride_2_ridgecv_lfp_threefold_baseline-1",
    "../../results/inter_area/monkeyF_v4_target_array_11_stride_2_ridgecv_lfp_threefold_baseline-1",
    "../../results/inter_area/monkeyF_v4_target_array_12_stride_2_ridgecv_lfp_threefold_baseline-1",
]

rundirs_v4_it_explore = [
    "../../results/inter_area/monkeyF_v4_target_array_9_stride_3_ridgecv_lfp_threefold_TESTS_baseline_0_dprime_-1",
    "../../results/inter_area/monkeyF_v4_target_array_9_stride_3_ridgecv_lfp_threefold_TESTS_baseline_0_dprime_1.5",
    "../../results/inter_area/monkeyF_v4_target_array_9_stride_3_ridgecv_lfp_threefold_TESTS_baseline_1_dprime_-1",
    "../../results/inter_area/monkeyF_v4_target_array_9_stride_3_ridgecv_lfp_threefold_TESTS_baseline_1_dprime_1.5",
    "../../results/inter_area/monkeyN_v4_target_array_13_stride_3_ridgecv_lfp_threefold_TESTS_baseline_0_dprime_-1",
    "../../results/inter_area/monkeyN_v4_target_array_13_stride_3_ridgecv_lfp_threefold_TESTS_baseline_0_dprime_1.5",
    "../../results/inter_area/monkeyN_v4_target_array_13_stride_3_ridgecv_lfp_threefold_TESTS_baseline_1_dprime_-1",
    "../../results/inter_area/monkeyN_v4_target_array_13_stride_3_ridgecv_lfp_threefold_TESTS_baseline_1_dprime_1.5"                          ]

rundirs_v1_v4 = rundirs_N_v1_v4 + rundirs_F_v1_v4
rundirs_v1_it = rundirs_N_v1_it + rundirs_F_v1_it
rundirs_v4_it = rundirs_N_v4_it + rundirs_F_v4_it


In [ ]:
run_group_select = "monkeyN v4 - it"
rectify = False

savedir = 'avg_surface_panels'
os.makedirs(savedir, exist_ok=True)

rundir_opts = {
    'monkeyF v1 - v4': rundirs_F_v1_v4,
    'monkeyF v1 - it': rundirs_F_v1_it,
    'monkeyF v4 - it': rundirs_F_v4_it,
    'v4 - it explore': rundirs_v4_it_explore,
    'monkeyN v1 - v4': rundirs_N_v1_v4,
    'monkeyN v1 - it': rundirs_N_v1_it,
    'monkeyN v4 - it': rundirs_N_v4_it,
    'v1 - v4': rundirs_v1_v4,
    'v1 - it': rundirs_v1_it,
    'v4 - it': rundirs_v4_it
}

rundirs = rundir_opts[run_group_select]


times_per_arr = []
delays_per_arr = []
granger_per_arr = []

for rundir in rundirs:
    print(rundir)
    all_success, all_complete, _, _ = check_run_integrity(rundir, njoblib=40)
    assert all_success and all_complete, f'run has not completed successfully. all_success {all_success} all_complete {all_complete}'
    inter_area_results, recurrent_results, params, cfg = load_inter_area_fit_chunks(rundir, with_tqdm=False)

    neg_recur = (recurrent_results < 0).sum()
    neg_inter = (inter_area_results < 0).sum()

    if neg_recur:
        print(f"{neg_recur} entries have negative recurrent performance")
        print(f"lowest performance: {recurrent_results.min()}")
        
    if neg_inter:
        print(f"{neg_inter} entries have negative inter-area performance")
        print(f"lowest performance: {inter_area_results.min()}")

    if rectify:
        recurrent_results[recurrent_results < 0] = 0
        inter_area_results[inter_area_results < 0] = 0
     
    granger_results = inter_area_results - recurrent_results
    
    assert not np.isnan(inter_area_results).any(), 'nan in inter-area results'
    assert not np.isnan(recurrent_results).any(), 'nan in recurrent results'
    assert not np.isnan(granger_results).any(), 'nan in granger results'
    
    granger_results = granger_results.mean(axis=1)  # avg over folds
    times = params[:,0]
    delays = params[:,1]

    print(granger_results.shape, times.shape, delays.shape)

    times_per_arr.append(times)
    delays_per_arr.append(delays)
    granger_per_arr.append(granger_results)


In [ ]:
# sort all results the same way

times_per_arr_srtd = []
delays_per_arr_srtd = []
granger_per_arr_srtd = []

for ts, ds, gs in zip(times_per_arr, delays_per_arr, granger_per_arr):
    idx = np.lexsort((ds, ts))
    ts = ts[idx]
    ds = ds[idx]
    gs = gs[idx]

    times_per_arr_srtd.append(ts)
    delays_per_arr_srtd.append(ds)
    granger_per_arr_srtd.append(gs)

In [ ]:
times_per_arr_srtd = np.array(times_per_arr_srtd)
delays_per_arr_srtd = np.array(delays_per_arr_srtd)

# assert each row has all equal values
t_diffs = times_per_arr_srtd - times_per_arr_srtd[0][None, :]
assert (t_diffs == 0).all()

d_diffs = delays_per_arr_srtd - delays_per_arr_srtd[0][None, :]
assert (d_diffs == 0).all()


all_granger_srtd = np.concatenate(granger_per_arr_srtd, axis=1)
all_granger_srtd.shape

# granger = all_granger_srtd.mean(axis=1)  # avg over all electrodes
# average over all arrays
granger = np.array([granger_arr.mean(axis=1) for granger_arr in granger_per_arr_srtd]).mean(axis=0)

times = times_per_arr_srtd[0]  # all equal, take first
delays = delays_per_arr_srtd[0]  # all equal, take first

source_times = times - delays
delays_to_target = times - source_times

plt.figure(figsize=HALF_PANEL_SIZE)
plt.tricontourf(source_times, delays_to_target, granger, levels=200)
plt.gca().set_aspect('equal', 'box')
plt.colorbar(shrink=.5, label=r'granger $r^2$')
plt.xlabel('source time (ms)')
plt.ylabel('delay to target (ms)')

# Compute avg. performance in a selected delay band

In [ ]:
delay_band = (10, 30)
unique_times = np.sort(np.unique(source_times))
unique_times = unique_times[(unique_times >= 0) & (unique_times <= 150)]

plt.figure(figsize=THIRD_PANEL_SIZE)
for i in range(len(rundirs)):
    granger_srtd_avg = granger_per_arr_srtd[i].mean(axis=1)
    
    avg_perfs_in_band = np.zeros_like(unique_times, dtype=float)
    avg_perfs_in_band.fill(np.nan)
    
    for i, t in enumerate(unique_times):
        mask = (source_times == t) & (delays_to_target >= delay_band[0]) & (delays_to_target <= delay_band[1])
        avg_perfs_in_band[i] = granger_srtd_avg[mask].mean()
    
    assert not np.isnan(avg_perfs_in_band).any(), 'did not fill array completely'
    
    plt.plot(unique_times, avg_perfs_in_band)
    
plt.xlabel("source time (ms)")
plt.ylabel(r"granger r^2")
plt.title(r"granger r^2 monkeyF V4 $\to$ IT avg. over arrays, delays 10-30ms")
plt.tight_layout()
plt.savefig('granger_v4_it_monkeyF_10-30ms_avg.svg')

In [ ]:

# compute best delay per time
def get_best_delay_per_time(times, delays, granger):
    ntimes = len(np.unique(times))

    best_delay_per_time = np.empty(ntimes)
    best_performance_per_time = np.empty(ntimes)

    best_delay_per_time.fill(np.nan)
    best_performance_per_time.fill(np.nan)

    unique_times = np.sort(np.unique(times))
    for i, t in enumerate(unique_times):
        mask = times == t
        idx_best_perf = np.argmax(granger[mask])
        best_perf = granger[mask][idx_best_perf]
        best_delay = delays[mask][idx_best_perf]

        best_delay_per_time[i] = best_delay
        best_performance_per_time[i] = best_perf

    # make sure whole array is filled
    assert not np.isnan(best_delay_per_time).any()
    assert not np.isnan(best_performance_per_time).any()
    return unique_times, best_performance_per_time, best_delay_per_time

ts, ps, ds = get_best_delay_per_time(times, delays, granger)
    

In [ ]:
# find peaks
peak_indices, peak_props = find_peaks(ps, prominence=.005)


peak_times = [ts[idx] for idx in peak_indices]
peak_perfs = [ps[idx] for idx in peak_indices]
peak_delays = [ds[idx] for idx in peak_indices]
peak_prominences = peak_props['prominences'].tolist()


plt.figure(figsize=THIRD_PANEL_SIZE)
plt.plot(ts, ps)
for peaktime in peak_times:
    plt.axvline(peaktime, color='green')
plt.xlabel('time in target array (ms)')
plt.ylabel(r'granger $r^2$')
plt.tight_layout()

# Potentially set manual time and delay markers

In [ ]:
SET_MANUAL = True

if SET_MANUAL:
    # -- F
    # peak_times = [74, 120]
    # peak_delays = [28, 40] 
    # -- N
    peak_times = [62, 112]
    peak_delays = [24, 36]

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def plot_surface(times, delays, granger, title, vmin=None, vmax=None, figsize=(HALF_WIDTH, HALF_WIDTH / 3), nticks=None, ticks=None, nolabels=False, text_offsetx=7, text_offsetfracy=4, cbarshrink=.7, xticks=None):
    
    fig = plt.figure(figsize=figsize)
    plt.tricontourf(times, delays, granger, levels=100, vmin=vmin, vmax=vmax)

    if vmin is None:
        vmin = granger.min()
    if vmax is None:
        vmax = granger.max()

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    mappable = cm.ScalarMappable(norm=norm)
    
    cb = plt.colorbar(mappable=mappable, shrink=cbarshrink, label=r'granger $r^2$', ticks=ticks, format=tkr.FormatStrFormatter('%.2f'), ax=plt.gca())
    if not nticks is None:
        cb.ax.locator_params(nbins=nticks)

    
    plt.gca().set_aspect('equal', 'box')
    if not nolabels:
        plt.title(title)
        
    # show diagonal for stim onset and stim offset
    min_delay = delays.min()
    max_delay = delays.max()
    
    stim_on_marker_x = [0 + min_delay, 0 + max_delay]
    stim_on_marker_y = [min_delay, max_delay]
    
    stim_off_marker_x = [250 + min_delay, 250 + max_delay]
    stim_off_marker_y = [min_delay, max_delay]
    
    plt.plot(stim_on_marker_x, stim_on_marker_y, color='white', linewidth=.5, linestyle=':')
    plt.text(max_delay / 4 - text_offsetx, max_delay / text_offsetfracy, s='stim. on', rotation=45, color='white', fontsize=5)
    
    plt.plot(stim_off_marker_x, stim_off_marker_y, color='white', linewidth=.5, linestyle=":")
    plt.text(250 + max_delay / 4 - text_offsetx, max_delay / text_offsetfracy, s='stim. off', rotation=45, color='white', fontsize=5)
    
    # make sure we do not draw out of bounds for offset
    plt.xlim((times.min(), times.max()))

    if not nolabels:
        plt.xlabel('time in target array (ms)')
        plt.ylabel('delay from source region (ms)')

    # set xticks
    if not xticks is None:
        plt.xticks(xticks, xticks)

    return fig


# ------------------------------------------------

# cbar ticks 
ticks = [
    0, (granger.max() - granger.min()) / 2 + granger.min(), granger.max()
]

# ------------------------------------------------

# xticks for surface
xticks = np.arange(0, 301, 100)

plot_surface(times, delays, granger, run_group_select + " (avg. over arrays)", ticks=ticks, figsize=(THIRD_WIDTH * 1.055, (THIRD_WIDTH / 2) * 1.055), text_offsetx=20, text_offsetfracy=6, cbarshrink=.5, xticks=xticks)

plt.plot(ts, ds, linewidth=.5, color='lightgrey', alpha=.5)

# plot peaks
plt.scatter(peak_times, peak_delays, s=2, linewidth=.3, edgecolor='white')

# clean up string to avoid problematic filenames and save
print(path.join(savedir, run_group_select.replace(' ', '_').replace('_-_', '-') + '.svg'))
plt.savefig(path.join(savedir, run_group_select.replace(' ', '_').replace('_-_', '-') + '.svg'))

print(peak_times, peak_delays)


In [ ]:
from matplotlib import ticker

MATCH_CMAPS = False
NOLABELS = True

# xticks for surface
xticks = np.arange(0, 301, 100)


def rundir_to_title(rundir):
    dirname = rundir.split('/')[-1]
    dirparts = dirname.split('_')
    monkey = dirparts[0]
    source_roi = dirparts[1]
    target_arr = dirparts[4]
    return f"{monkey} {source_roi} - array {target_arr}"


if MATCH_CMAPS:
    vmin = np.array([granger.mean(axis=1).min() for granger in granger_per_arr_srtd]).min()
    vmax = np.array([granger.mean(axis=1).max() for granger in granger_per_arr_srtd]).max()
    
    ticks = [
        0.,  # force tick at zero, often rounding otherwise puts a label on -0.0
        (vmax - vmin) / 2 + vmin,
        vmax
    ]

for i, (rundir, times, delays, granger) in enumerate(zip(rundirs, times_per_arr_srtd, delays_per_arr_srtd, granger_per_arr_srtd)):
    title = rundir_to_title(rundir)
    granger = granger.mean(axis=1)
    print(f"{i}: {rundir} {times.shape} {delays.shape} {granger.shape}")


    if not MATCH_CMAPS:
        vmin = 0.
        vmax = granger.max()

        print(vmin, vmax)
        
        ticks = [
            0.,  # force tick at zero, often rounding otherwise puts a label on -0.0
            (vmax - vmin) / 2 + vmin,
            vmax
        ]

    plot_surface(times, delays, granger, title, figsize=(THIRD_WIDTH * 1.055, (THIRD_WIDTH / 2) * 1.055), ticks=ticks, vmin=vmin, vmax=vmax, nolabels=True, text_offsetx=20, text_offsetfracy=6, cbarshrink=.5, xticks=xticks)
    # plt.gca().set_title(rundir, fontsize=4)
    
    # plot peaks (based on avg. surface!)
    plt.scatter(peak_times, peak_delays, s=2, linewidth=.3, edgecolor='white')

    plt.savefig(path.join(savedir, title.replace(' ', '_').replace('_-_', '-') + '.svg'))

In [ ]:
savedir

# OLD

In [ ]:
from scipy.stats import rankdata
import numpy as np
from skimage.feature import peak_local_max

# create an array
unique_delays = np.unique(delays)
unique_times = np.unique(times)

delay_idx = rankdata(delays, method='dense') - 1
time_idx = rankdata(times, method='dense') - 1

surf_arr = np.zeros((len(unique_delays), len(unique_times)))
surf_arr[delay_idx, time_idx] = granger

plt.figure(figsize=HALF_PANEL_SIZE)
plt.imshow(surf_arr, origin='lower')

peaks = peak_local_max(surf_arr, min_distance=5)
peak_x = peaks[:, 1]
peak_y = peaks[:, 0]

plt.scatter(peak_x, peak_y, s=3)